In [385]:
# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp
from scipy.stats import randint, uniform

# 머신러닝 
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score, balanced_accuracy_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier, LGBMRegressor


# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [ ]:
# # 1. 데이터 로드
# df = pd.read_csv('data/merged_final_data.csv')

# df['freight_delay_penalty'] = df['freight_value'] * df['delay_days']
# df['price_delay_penalty'] = df['price'] * df['delay_days']
# df['total_delay_penalty'] = df['total_price'] * df['delay_days']
# df['price_per_item'] = df['total_price'] / (df['order_item_id'] + 0.1) # 0 나누기 에러 방지
# df['freight_distance_interaction'] = df['freight_value'] * df['distance_km']
# df['delivery_freight_cost'] = df['delivery_days'] * df['freight_value']
# df['is_weekend_purchase'] = df['order_purchase_dayofweek'].isin(['saturday', 'sunday']).astype(int)

# # 2. 불필요한 컬럼 제거 (ID, 시간, 타겟과 겹치는 변수 등)
# drop_cols = [
#     'customer_unique_id', 'customer_city', 'seller_city', 'shipping_limit_date'
# ]
# df_processed = df.drop(columns=drop_cols, errors='ignore')

# # 3. 범주형 변수 원핫 인코딩 
# df_encoded = pd.get_dummies(df_processed)

# # 4. 독립변수(X)와 종속변수(y) 분리
# X = df_encoded.drop(columns=['review_score'])
# y = df_encoded['review_score']

# # 5. 학습용/테스트용 데이터 분리 (8:2 비율)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


# # import optuna
# # from xgboost import XGBRegressor
# # from sklearn.metrics import mean_squared_error
# # import numpy as np

# # # 1. Optuna 목적 함수(Objective Function) 정의
# # # 이 함수 안에서 Optuna가 파라미터를 요리조리 바꿔가며 테스트를 진행합니다.
# # def objective(trial):
# #     # 튜닝할 하이퍼파라미터의 탐색 범위 설정 (과적합 방지에 초점)
# #     param = {
# #         'n_estimators': 3000, # 최대 트리는 넉넉하게 잡고 조기 종료로 제어
# #         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
# #         'max_depth': trial.suggest_int('max_depth', 3, 8), # 너무 깊어지지 않도록 3~8로 제한
# #         'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
# #         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
# #         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
# #         'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),  # L1 정규화 (피처 솎아내기)
# #         'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True), # L2 정규화 (가중치 누르기)
# #         'tree_method': 'hist', # M4 Pro의 빠른 연산력 활용
# #         'random_state': 42,
# #         'n_jobs': -1,
# #         'early_stopping_rounds': 50,
# #         'eval_metric': 'rmse'
# #     }

# #     # 제안된 파라미터로 모델 생성
# #     model = XGBRegressor(**param)
    
# #     # 모델 학습 (조기 종료 적용)
# #     model.fit(
# #         X_train, y_train,
# #         eval_set=[(X_valid, y_valid)],
# #         verbose=False # 튜닝 중에는 로그가 너무 많아지므로 숨김 처리
# #     )
    
# #     # 검증 데이터(Validation)로 예측 후 RMSE 계산
# #     preds = np.clip(model.predict(X_valid), 1, 5)
# #     rmse = np.sqrt(mean_squared_error(y_valid, preds))
    
# #     # Optuna는 이 반환된 RMSE 값이 '최소화'되는 방향으로 파라미터를 찾아갑니다.
# #     return rmse

# # # 2. Optuna Study 생성 및 실행
# # print("하이퍼파라미터 튜닝을 시작합니다... (M4 Pro 풀가동 🚀)")
# # study = optuna.create_study(direction='minimize') # 목표: RMSE 최소화

# # # n_trials: 몇 번의 파라미터 조합을 테스트해 볼 것인지 설정 (시간 여유에 따라 50~100 추천)
# # study.optimize(objective, n_trials=50) 

# # # 3. 최적의 파라미터 결과 출력
# # print("\n=== 튜닝 완료! 가장 성능이 좋았던 Best 파라미터 ===")
# # best_params = study.best_trial.params
# # for key, value in best_params.items():
# #     print(f"    '{key}': {value},")

# # print(f"\n이때의 Validation RMSE: {study.best_trial.value:.4f}")

# # best 파라미터 값
# # learning_rate=0.03,      
# #     max_depth=3,
# #     min_child_weight=3,             
# #     subsample=0.767,
# #     colsample_bytree=0.844,
# #     reg_alpha=0.029,
# #     reg_lambda=6.716,

# # 5. XGBoost 회귀 모델 생성
# xgb_model = XGBRegressor(
#     n_estimators=3000,        
#     learning_rate=0.03,      
#     max_depth=3,
#     min_child_weight=3,             
#     subsample=0.767,
#     colsample_bytree=0.844,
#     reg_alpha=0.029,
#     reg_lambda=6.716,
#     tree_method='hist',      
#     n_jobs=-1,               
#     random_state=42,
#     eval_metric='rmse'  
# )

# # 모델 학습 진행
# print("모델 학습을 시작합니다... (러닝 커브 기록 중)")
# xgb_model.fit(
#     X_train, y_train,
#     eval_set=[(X_train, y_train), (X_valid, y_valid)],
#     verbose=100                                     
# )
# print("학습 완료!")

# # 7. 예측 및 성능 평가
# y_pred_valid = xgb_model.predict(X_valid)
# y_pred = xgb_model.predict(X_test)

# # 리뷰 스코어 클리핑(Clipping) 처리
# y_pred_valid = np.clip(y_pred_valid, 1, 5)
# y_pred = np.clip(y_pred, 1, 5)

# rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
# mae_valid = mean_absolute_error(y_valid, y_pred_valid)
# r2_valid = r2_score(y_valid, y_pred_valid)

# print("\n=== Validation 평가 지표 ===")
# print(f"Validation RMSE: {rmse_valid:.4f}")
# print(f"Validation MAE: {mae_valid:.4f}")
# print(f"Validation R2 Score: {r2_valid:.4f}")

# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# mae = mean_absolute_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# print("\n=== 최종 Test 모델 평가 지표 ===")
# print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
# print(f"MAE (평균 절대 오차): {mae:.4f}")
# print(f"R2 Score (결정계수): {r2:.4f}")

# # 8. 러닝 커브 시각화 코드 추가
# results = xgb_model.evals_result()
# epochs = len(results['validation_0']['rmse'])
# x_axis = range(0, epochs)

# plt.figure(figsize=(10, 6))

# # validation_0은 Train, validation_1은 Validation 데이터입니다.
# plt.plot(x_axis, results['validation_0']['rmse'], label='Train RMSE', color='blue', alpha=0.7)
# plt.plot(x_axis, results['validation_1']['rmse'], label='Validation RMSE', color='red', alpha=0.7)

# plt.title('XGBoost Learning Curve (Train vs Validation RMSE)', fontsize=16, fontweight='bold')
# plt.xlabel('Number of Trees (n_estimators)', fontsize=12)
# plt.ylabel('RMSE', fontsize=12)
# plt.legend(fontsize=12)
# plt.grid(True, linestyle='--', alpha=0.6)

# plt.tight_layout()
# plt.show()

In [386]:
df = pd.read_csv('data/merged_final_data.csv')

In [387]:
cols = df.columns
cols 

Index(['order_id', 'customer_id', 'customer_unique_id', 'customer_city',
       'customer_state', 'order_item_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'review_score', 'category', 'seller_city',
       'seller_state', 'order_purchase_dayofweek', 'order_purchase_month',
       'approved_days', 'dispatch_days', 'delivery_days',
       'expected_delivery_days', 'delay_days', 'delay_days_int', 'is_delayed',
       'delay_days_cat', 'main_category', 'sub_category', 'distance_km',
       'distance_cat', 'cross_state'],
      dtype='str')

In [388]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 109294 entries, 0 to 109293
Data columns (total 29 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  109294 non-null  str    
 1   customer_id               109294 non-null  str    
 2   customer_unique_id        109294 non-null  str    
 3   customer_city             109294 non-null  str    
 4   customer_state            109294 non-null  str    
 5   order_item_id             109294 non-null  int64  
 6   seller_id                 109294 non-null  str    
 7   shipping_limit_date       109294 non-null  str    
 8   price                     109294 non-null  float64
 9   freight_value             109294 non-null  float64
 10  review_score              109294 non-null  int64  
 11  category                  107749 non-null  str    
 12  seller_city               109294 non-null  str    
 13  seller_state              109294 non-null  str    
 14 

In [389]:
df.isna().sum()

order_id                       0
customer_id                    0
customer_unique_id             0
customer_city                  0
customer_state                 0
order_item_id                  0
seller_id                      0
shipping_limit_date            0
price                          0
freight_value                  0
review_score                   0
category                    1545
seller_city                    0
seller_state                   0
order_purchase_dayofweek       0
order_purchase_month           0
approved_days                  0
dispatch_days                  0
delivery_days                  0
expected_delivery_days         0
delay_days                     0
delay_days_int                 0
is_delayed                     0
delay_days_cat                 0
main_category                  0
sub_category                   0
distance_km                    0
distance_cat                   0
cross_state                    0
dtype: int64

### 머신러닝 모델 생성시 절대 안쓸만한 컬럼 제거

In [390]:
cols_to_drop = ['order_id','customer_id','seller_id','category','delay_days_int']
df = df.drop(columns=cols_to_drop)
df['delay_days'] = df['delay_days'].astype(int) # delay_days 정수화

In [391]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 109294 entries, 0 to 109293
Data columns (total 24 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_unique_id        109294 non-null  str    
 1   customer_city             109294 non-null  str    
 2   customer_state            109294 non-null  str    
 3   order_item_id             109294 non-null  int64  
 4   shipping_limit_date       109294 non-null  str    
 5   price                     109294 non-null  float64
 6   freight_value             109294 non-null  float64
 7   review_score              109294 non-null  int64  
 8   seller_city               109294 non-null  str    
 9   seller_state              109294 non-null  str    
 10  order_purchase_dayofweek  109294 non-null  str    
 11  order_purchase_month      109294 non-null  int64  
 12  approved_days             109294 non-null  int64  
 13  dispatch_days             109294 non-null  int64  
 14 

In [392]:
df.isna().sum()

customer_unique_id          0
customer_city               0
customer_state              0
order_item_id               0
shipping_limit_date         0
price                       0
freight_value               0
review_score                0
seller_city                 0
seller_state                0
order_purchase_dayofweek    0
order_purchase_month        0
approved_days               0
dispatch_days               0
delivery_days               0
expected_delivery_days      0
delay_days                  0
is_delayed                  0
delay_days_cat              0
main_category               0
sub_category                0
distance_km                 0
distance_cat                0
cross_state                 0
dtype: int64

In [393]:
df.to_csv('./data/ml_data.csv')

In [394]:
df2 = df.copy()
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 109294 entries, 0 to 109293
Data columns (total 24 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_unique_id        109294 non-null  str    
 1   customer_city             109294 non-null  str    
 2   customer_state            109294 non-null  str    
 3   order_item_id             109294 non-null  int64  
 4   shipping_limit_date       109294 non-null  str    
 5   price                     109294 non-null  float64
 6   freight_value             109294 non-null  float64
 7   review_score              109294 non-null  int64  
 8   seller_city               109294 non-null  str    
 9   seller_state              109294 non-null  str    
 10  order_purchase_dayofweek  109294 non-null  str    
 11  order_purchase_month      109294 non-null  int64  
 12  approved_days             109294 non-null  int64  
 13  dispatch_days             109294 non-null  int64  
 14 

In [395]:
# feature engineering

df2 = df.copy()

# 베송속도 관련 파생피처들
df2['delivery_speed'] = df2['distance_km'] / df2['delivery_days']
df2['day_per_km'] = df2['delivery_days'] / df2['distance_km']
df2['delivery_ratio'] = df2['delivery_days'] / df2['expected_delivery_days']

# 가격 대비 배송비
df2['freight_ratio'] = df2['freight_value'] / df2['price']

# delivery_days * distance_km
df2["delivery_distance"] = df2["delivery_days"] * df2["distance_km"]

# delivery_days * price
df2["delivery_price"] = (df2["delivery_days"] * df2["price"])

# price + freight_value
df2['total_price'] = df2['price'] + df2['freight_value']

df2 = df2.replace([np.inf, -np.inf], np.nan)
num_cols = df2.select_dtypes(include=[np.number]).columns
df2[num_cols] = df2[num_cols].fillna(0)

In [396]:
df2.isna().sum()

customer_unique_id          0
customer_city               0
customer_state              0
order_item_id               0
shipping_limit_date         0
price                       0
freight_value               0
review_score                0
seller_city                 0
seller_state                0
order_purchase_dayofweek    0
order_purchase_month        0
approved_days               0
dispatch_days               0
delivery_days               0
expected_delivery_days      0
delay_days                  0
is_delayed                  0
delay_days_cat              0
main_category               0
sub_category                0
distance_km                 0
distance_cat                0
cross_state                 0
delivery_speed              0
day_per_km                  0
delivery_ratio              0
freight_ratio               0
delivery_distance           0
delivery_price              0
total_price                 0
dtype: int64

In [398]:
# 브라질 물류 특성을 반영한 경계값 설정
# 0-50(도시내), 50-250(인접도시), 250-750(핵심노선), 750-1500(지역간), 1500+(초장거리)
bins = [0, 50, 250, 750, 1500, np.inf]
labels = [0, 1, 2, 3, 4]

df2['distance_cat'] = pd.cut(
    df2['distance_km'], 
    bins=bins, 
    labels=labels, 
    include_lowest=True
).astype(int)

# 각 구간에 데이터가 골고루 분포되었는지 확인
print(df2['distance_cat'].value_counts().sort_index())

distance_cat
0    13382
1    17689
2    48704
3    19790
4     9729
Name: count, dtype: int64


In [399]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 109294 entries, 0 to 109293
Data columns (total 33 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_unique_id        109294 non-null  str    
 1   customer_city             109294 non-null  str    
 2   customer_state            109294 non-null  str    
 3   order_item_id             109294 non-null  int64  
 4   shipping_limit_date       109294 non-null  str    
 5   price                     109294 non-null  float64
 6   freight_value             109294 non-null  float64
 7   review_score              109294 non-null  int64  
 8   seller_city               109294 non-null  str    
 9   seller_state              109294 non-null  str    
 10  order_purchase_dayofweek  109294 non-null  str    
 11  order_purchase_month      109294 non-null  int64  
 12  approved_days             109294 non-null  int64  
 13  dispatch_days             109294 non-null  int64  
 14 

In [400]:
### 상파울루인지 구분하는 컬럼
# 기본값을 0으로 설정 (둘 다 SP가 아닌 경우: 일반/장거리 노선)
df2['sp_route_type'] = 0

# 조건 1: 한 쪽이라도 SP인 경우 (부분적 인프라 혜택)
cond_partial = (df2['seller_state'] == 'SP') | (df2['customer_state'] == 'SP')
df2.loc[cond_partial, 'sp_route_type'] = 1

# 조건 2: 둘 다 SP인 경우 (최적화된 인프라, 라스트마일 최적화)
cond_internal = (df2['seller_state'] == 'SP') & (df2['customer_state'] == 'SP')
df2.loc[cond_internal, 'sp_route_type'] = 2



### 상파울루 고객 여부 (True/False를 1/0으로 변환)
df2['is_sp_customer'] = (df2['customer_state'] == 'SP').astype(int)

# 기본값을 0으로 설정 (둘 다 SP가 아닌 경우: 일반/장거리 노선 및 고객이 SP가 아닌 경우)
df2['sp_route_type_customer'] = 0

# 조건 1: 고객이 SP인 경우 (부분적 인프라 혜택)
cond_partial = (df2['customer_state'] == 'SP')
df2.loc[cond_partial, 'sp_route_type_customer'] = 1

# 조건 2: 둘 다 SP인 경우 (최적화된 인프라, 라스트마일 최적화)
cond_internal = (df2['seller_state'] == 'SP') & (df2['customer_state'] == 'SP')
df2.loc[cond_internal, 'sp_route_type_customer'] = 2

# 결과 확인
print(df2['sp_route_type_customer'].value_counts().sort_index())



### 상파울루 셀러 여부 (True/False를 1/0으로 변환)
df2['is_sp_seller'] = (df2['seller_state'] == 'SP').astype(int)

# 기본값을 0으로 설정 (둘 다 SP가 아닌 경우: 일반/장거리 노선 및 셀러가 SP가 아닌 경우)
df2['sp_route_type_seller'] = 0

# 조건 1: 셀러가 SP인 경우 (부분적 인프라 혜택)
cond_partial_seller = (df2['seller_state'] == 'SP')
df2.loc[cond_partial_seller, 'sp_route_type_seller'] = 1

# 조건 2: 둘 다 SP인 경우 (최적화된 인프라, 라스트마일 최적화)
cond_internal_seller = (df2['seller_state'] == 'SP') & (df2['customer_state'] == 'SP')
df2.loc[cond_internal_seller, 'sp_route_type_seller'] = 2

# 결과 확인
print(df2['sp_route_type_seller'].value_counts().sort_index())

sp_route_type_customer
0    63171
1    10959
2    35164
Name: count, dtype: int64
sp_route_type_seller
0    31348
1    42782
2    35164
Name: count, dtype: int64


# 회귀모델

In [401]:
X = df2.drop(columns=['review_score','customer_unique_id', 'customer_city', 'seller_city', 'shipping_limit_date'])
y = df2['review_score']

In [402]:
# 1. 원본 데이터 상태에서 분리 (원핫 인코딩 수행 전)
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid, test_size=0.2, random_state=42, stratify=y_train_valid
)

# 2. 수치형/범주형 컬럼 분리
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

# 3. 전처리 파이프라인 정의
numeric_transformer = SimpleImputer(strategy='median') # 수치형은 중앙값으로 결측치 대체
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # 범주형은 최빈값으로 대체
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # 새로운 범주 등장 시 무시
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

### 1번째 회귀모델 : XGB Regressor

In [403]:
# 1. XGBoost 회귀 모델을 포함한 파이프라인 정의
# (분류기가 아니므로 이름을 'regressor'로 변경합니다)
xgb_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # 이전에 만든 전처리기 그대로 재사용
    ('regressor', XGBRegressor(
        n_estimators=1300, 
        learning_rate=0.03078,
        max_depth=4,
        min_child_weight=20,
        reg_lambda=3.0,
        tree_method='hist',
        subsample=0.87330,           # 데이터 샘플링 비율 (약 87%)
        colsample_bytree=0.60638,    # 피처 샘플링 비율 (약 60%)
        n_jobs=-1,
        random_state=42
    ))
])

In [404]:
# # 2. RandomizedSearchCV를 위한 탐색 공간 정의
# # (접두사가 'regressor__'로 변경된 것에 주의)
# xgb_param_dist = {
#     'regressor__n_estimators': [500, 700, 900], 
#     'regressor__learning_rate': uniform(0.01, 0.05),    # 0.01 ~ 0.10 사이의 무작위 실수
#     'regressor__max_depth': [4, 5, 6, 7],             # 5 ~ 11 사이의 무작위 정수
#     'regressor__min_child_weight': randint(10, 30),
#     'regressor__subsample': uniform(0.6, 0.3),          # 0.6 ~ 1.0 사이 무작위 비율
#     'regressor__colsample_bytree': uniform(0.6, 0.3),    # 0.6 ~ 1.0 사이 무작위 비율
#     'regressor__reg_lambda': uniform(1.0, 4.0)
# }

# # 3. RandomizedSearchCV 객체 생성
# xgb_random_search = RandomizedSearchCV(
#     estimator=xgb_reg_pipeline,
#     param_distributions=xgb_param_dist,
#     n_iter=20, 
#     cv=3,
#     # ★ 회귀 모델의 핵심: 오차(RMSE)가 작을수록 좋으므로 'neg_root_mean_squared_error' 사용
#     scoring='neg_root_mean_squared_error', 
#     random_state=42,
#     n_jobs=-1,
#     verbose=1
# )

# # 4. 하이퍼파라미터 튜닝 진행
# print("회귀 모델 하이퍼파라미터 탐색을 시작합니다...")
# xgb_random_search.fit(X_train, y_train)
# print("탐색 완료!\n")

# # 5. 최적의 파라미터 확인
# print("=== Best Hyperparameters ===")
# print(xgb_random_search.best_params_)

# # === Best Hyperparameters ===
# # {'regressor__colsample_bytree': np.float64(0.7796596399465607), 'regressor__learning_rate': np.float64(0.04473924665198523), 'regressor__max_depth': 7, 'regressor__min_child_weight': 11, 'regressor__n_estimators': 700, 'regressor__reg_lambda': np.float64(4.232481518257668), 'regressor__subsample': np.float64(0.790021126953127)}

In [405]:
# 6. 파이프라인 학습 (내부적으로 X_train만 전처리 규칙을 학습)
xgb_reg_pipeline.fit(X_train, y_train)

# 7. 검증(Valid) 및 테스트(Test) 세트 예측
y_pred_valid = xgb_reg_pipeline.predict(X_valid)

In [406]:
# 8. 클리핑(Clipping) 처리 
# (회귀 모델은 0.5점이나 5.8점 같은 범위를 벗어난 값을 뱉을 수 있으므로 1~5로 고정)
y_pred_valid = np.clip(y_pred_valid, 1, 5)

# 9. 최종 성능 평가 지표 출력
rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
r2_valid = r2_score(y_valid, y_pred_valid)

print("\n=== Validation 세트 평가 지표 ===")
print(f"RMSE: {rmse_valid:.4f} | MAE: {mae_valid:.4f} | R2 Score: {r2_valid:.4f}")



=== Validation 세트 평가 지표 ===
RMSE: 1.1904 | MAE: 0.9211 | R2 Score: 0.2187


In [409]:
# [Step 1] 데이터 전처리 (Pipeline을 쓰지 않으므로 수동으로 변환)
X_train_transformed = preprocessor.fit_transform(X_train)
X_valid_transformed = preprocessor.transform(X_valid)

# 7. XGBoost 모델 설정 (M4 Pro 최적화)
xgb_model = XGBRegressor(
    n_estimators=1300, 
    learning_rate=0.03078,
    max_depth=4,
    min_child_weight=20,
    reg_lambda=3.0,
    tree_method='hist',
    subsample=0.87330,           # 데이터 샘플링 비율 (약 87%)
    colsample_bytree=0.60638,    # 피처 샘플링 비율 (약 60%)
    n_jobs=-1,
    random_state=42,
    early_stopping_rounds=50
)

# 8. 모델 학습 (Valid 세트를 활용해 실시간으로 평가)
print("XGBoost 회귀 모델 학습을 시작합니다...")
xgb_model.fit(
    X_train_transformed, 
    y_train,
    eval_set=[(X_valid_transformed, y_valid)], # 검증 데이터로 모니터링
    verbose=0                                 
)

# [Step 4] 예측 및 클리핑(Clipping) 처리
y_pred_valid = xgb_model.predict(X_valid_transformed)
# 점수 범위가 1~5일 경우 범위를 벗어난 값을 고정해줍니다.
y_pred_valid = np.clip(y_pred_valid, 1, 5)

# [Step 5] 최종 성능 평가
rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
r2_valid = r2_score(y_valid, y_pred_valid)

print("\n=== 최종 평가 결과 (Validation) ===")
print(f"최적의 반복 횟수(Best Iteration): {xgb_model.best_iteration}")
print(f"RMSE: {rmse_valid:.4f}")
print(f"MAE: {mae_valid:.4f}")
print(f"R2 Score: {r2_valid:.4f}")

XGBoost 회귀 모델 학습을 시작합니다...

=== 최종 평가 결과 (Validation) ===
최적의 반복 횟수(Best Iteration): 1299
RMSE: 1.1904
MAE: 0.9211
R2 Score: 0.2187


In [ ]:
# 피처 중요도 확인
actual_feature_names = preprocessor.get_feature_names_out() 

imp = pd.Series(
    xgb_model.feature_importances_,
    index=actual_feature_names
).sort_values(ascending=False)

print(imp.head(50))

num__is_delayed                                 0.363016
cat__delay_days_cat_1-3일 지연                     0.066318
num__delay_days                                 0.060466
cat__delay_days_cat_조기                          0.047293
num__delivery_ratio                             0.034960
num__order_item_id                              0.032905
cat__delay_days_cat_10일 이상 지연                   0.024632
cat__delay_days_cat_4-6일 지연                     0.013827
num__delivery_days                              0.012859
cat__sub_category_Furniture & Decor             0.012352
cat__main_category_Home & Living                0.010404
cat__main_category_Electronics & Tech           0.007656
num__cross_state                                0.006290
cat__sub_category_Books & Media                 0.005953
cat__sub_category_IT & Computers                0.005378
cat__sub_category_Telephony                     0.005291
num__dispatch_days                              0.005258
cat__customer_state_RJ         

### 2번째 회귀모델 : LGBM Regressor

In [411]:
# 1. 파이프라인 정의
lgbm_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LGBMRegressor(
        colsample_bytree=0.7794,
        learning_rate=0.0661,
        max_depth=8,
        min_child_samples=33,
        n_estimators=1000,
        num_leaves=32,
        random_state=42, 
        verbose=-1, 
        subsample=0.8689,
        n_jobs=-1))
])


In [412]:
# # 2. 탐색 공간 (LGBM 전용)
# lgbm_param_dist = {
#     'regressor__n_estimators': [500, 800, 1000],
#     'regressor__learning_rate': uniform(0.02, 0.05),
#     'regressor__num_leaves': randint(25, 45),         # 트리의 복잡도 (가장 중요)
#     'regressor__max_depth': [5, 6, 7, 8],        # -1은 무제한
#     'regressor__min_child_samples': randint(20, 40),  # 과적합 방지
#     'regressor__colsample_bytree': uniform(0.6, 0.3),
#     'regressor__subsample': uniform(0.7, 0.2)
# }

# # 3. RandomizedSearchCV
# lgbm_random_search = RandomizedSearchCV(
#     estimator=lgbm_reg_pipeline,
#     param_distributions=lgbm_param_dist,
#     n_iter=20, 
#     cv=3,
#     scoring='neg_root_mean_squared_error',
#     random_state=42,
#     n_jobs=-1,
#     verbose=1
# )

# print("LightGBM 회귀 튜닝 시작...")
# lgbm_random_search.fit(X_train, y_train)
# print("Best LGBM Params:", lgbm_random_search.best_params_)

# # Best LGBM Params: {'regressor__colsample_bytree': np.float64(0.7793699936433255), 'regressor__learning_rate': np.float64(0.06609371175115585), 'regressor__max_depth': 8, 'regressor__min_child_samples': 33, 'regressor__n_estimators': 1000, 'regressor__num_leaves': 32, 'regressor__subsample': np.float64(0.8689067697356303)}

In [413]:
# 6. 파이프라인 학습 (내부적으로 X_train만 전처리 규칙을 학습)
lgbm_reg_pipeline.fit(X_train, y_train)

# 7. 검증(Valid) 및 테스트(Test) 세트 예측
y_pred_valid = lgbm_reg_pipeline.predict(X_valid)

In [414]:
# 8. 클리핑(Clipping) 처리 
# (회귀 모델은 0.5점이나 5.8점 같은 범위를 벗어난 값을 뱉을 수 있으므로 1~5로 고정)
y_pred_valid = np.clip(y_pred_valid, 1, 5)

# 9. 최종 성능 평가 지표 출력
rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
r2_valid = r2_score(y_valid, y_pred_valid)

print("\n=== Validation 세트 평가 지표 ===")
print(f"RMSE: {rmse_valid:.4f} | MAE: {mae_valid:.4f} | R2 Score: {r2_valid:.4f}")



=== Validation 세트 평가 지표 ===
RMSE: 1.1719 | MAE: 0.8997 | R2 Score: 0.2429


### 3번째 회귀모델 : Random Forest Regressor

In [415]:
# 1. 파이프라인 정의
rf_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        max_depth=15, 
        max_features='log2',
        min_samples_leaf=15,
        min_samples_split=3,
        n_estimators=300,
        random_state=42, 
        n_jobs=-1))
])

In [416]:
# # 2. 탐색 공간 (RF 전용)
# rf_param_dist = {
#     'regressor__n_estimators': [100, 300, 500],       # RF는 나무가 너무 많으면 매우 느려지니 주의
#     'regressor__max_depth': [10, 20, 30, None],       # None은 끝까지 깊게
#     'regressor__min_samples_split': randint(2, 11),   # 노드를 나누기 위한 최소 샘플 수
#     'regressor__min_samples_leaf': randint(1, 11),    # 리프 노드가 되기 위한 최소 샘플 수
#     'regressor__max_features': ['sqrt', 'log2', None] # 피처 선택 방식
# }

# # 3. RandomizedSearchCV
# rf_random_search = RandomizedSearchCV(
#     estimator=rf_reg_pipeline,
#     param_distributions=rf_param_dist,
#     n_iter=15, # RF는 연산량이 많으므로 n_iter를 조금 줄여서 시작하세요.
#     cv=3,
#     scoring='neg_root_mean_squared_error',
#     random_state=42,
#     n_jobs=-1,
#     verbose=1
# )

# print("Random Forest 회귀 튜닝 시작...")
# rf_random_search.fit(X_train, y_train)
# print("Best RF Params:", rf_random_search.best_params_)

# # Fitting 3 folds for each of 15 candidates, totalling 45 fits
# # Best RF Params: {'regressor__max_depth': None, 'regressor__max_features': 'log2', 
# # 'regressor__min_samples_leaf': 2, 'regressor__min_samples_split': 3, 'regressor__n_estimators': 300}

In [417]:
# 6. 파이프라인 학습 (내부적으로 X_train만 전처리 규칙을 학습)
rf_reg_pipeline.fit(X_train, y_train)

# 7. 검증(Valid) 및 테스트(Test) 세트 예측
y_pred_valid = rf_reg_pipeline.predict(X_valid)

In [418]:
# 8. 클리핑(Clipping) 처리 
# (회귀 모델은 0.5점이나 5.8점 같은 범위를 벗어난 값을 뱉을 수 있으므로 1~5로 고정)
y_pred_valid = np.clip(y_pred_valid, 1, 5)

# 9. 최종 성능 평가 지표 출력
rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
r2_valid = r2_score(y_valid, y_pred_valid)

print("\n=== Validation 세트 평가 지표 ===")
print(f"RMSE: {rmse_valid:.4f} | MAE: {mae_valid:.4f} | R2 Score: {r2_valid:.4f}")



=== Validation 세트 평가 지표 ===
RMSE: 1.2044 | MAE: 0.9409 | R2 Score: 0.2003


In [419]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def check_overfitting(model, X_train, y_train, X_valid, y_valid, model_name="Model"):
    # 1. Train, Valid 예측
    y_train_pred = model.predict(X_train)
    y_valid_pred = model.predict(X_valid)

    # 2. 클리핑 처리 (1~5점 범위를 벗어나는 값 보정, 작성하신 코드와 동일)
    y_train_pred = np.clip(y_train_pred, 1, 5)
    y_valid_pred = np.clip(y_valid_pred, 1, 5)

    # 3. 평가지표 계산
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    valid_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred))
    
    train_mae = mean_absolute_error(y_train, y_train_pred)
    valid_mae = mean_absolute_error(y_valid, y_valid_pred)
    
    train_r2 = r2_score(y_train, y_train_pred)
    valid_r2 = r2_score(y_valid, y_valid_pred)

    # 4. 결과 출력
    print(f"[{model_name}] 과적합(Overfitting) 진단 결과")
    print("-" * 55)
    print(f"{'Metric':<10} | {'Train Score':<12} | {'Valid Score':<12} | {'Diff (Train - Valid)'}")
    print("-" * 55)
    # RMSE와 MAE는 낮을수록 좋으므로 Train이 더 낮으면 음수가 됨
    print(f"{'RMSE':<10} | {train_rmse:<12.4f} | {valid_rmse:<12.4f} | {train_rmse - valid_rmse:+.4f}")
    print(f"{'MAE':<10} | {train_mae:<12.4f} | {valid_mae:<12.4f} | {train_mae - valid_mae:+.4f}")
    # R2는 높을수록 좋으므로 Train이 더 높으면 양수가 됨
    print(f"{'R2 Score':<10} | {train_r2:<12.4f} | {valid_r2:<12.4f} | {train_r2 - valid_r2:+.4f}")
    print("-" * 55)
    print()

# ---------------------------------------------------------
# 모델별 과적합 확인 실행 (위에서 학습한 파이프라인 3개 적용)
# ---------------------------------------------------------
check_overfitting(xgb_reg_pipeline, X_train, y_train, X_valid, y_valid, "1. XGBoost")
check_overfitting(lgbm_reg_pipeline, X_train, y_train, X_valid, y_valid, "2. LightGBM")
check_overfitting(rf_reg_pipeline, X_train, y_train, X_valid, y_valid, "3. Random Forest")

[1. XGBoost] 과적합(Overfitting) 진단 결과
-------------------------------------------------------
Metric     | Train Score  | Valid Score  | Diff (Train - Valid)
-------------------------------------------------------
RMSE       | 1.1500       | 1.1904       | -0.0404
MAE        | 0.8903       | 0.9211       | -0.0308
R2 Score   | 0.2709       | 0.2187       | +0.0522
-------------------------------------------------------

[2. LightGBM] 과적합(Overfitting) 진단 결과
-------------------------------------------------------
Metric     | Train Score  | Valid Score  | Diff (Train - Valid)
-------------------------------------------------------
RMSE       | 1.0123       | 1.1719       | -0.1595
MAE        | 0.7759       | 0.8997       | -0.1238
R2 Score   | 0.4351       | 0.2429       | +0.1922
-------------------------------------------------------

[3. Random Forest] 과적합(Overfitting) 진단 결과
-------------------------------------------------------
Metric     | Train Score  | Valid Score  | Diff (Train - 